# Notebook 3 — Level 3 MONITORING: ViT Damage Classification
## Transfer Learning for Post-Impact Damage Assessment

**Model:** `google/vit-base-patch16-224`  
**Class map:** D ∈ {0=no damage, 1=microcracks, 2=local fragmentation, 3=penetration}

---

**Outputs saved for downstream notebooks:**
- `models/vit_damage/` — fine-tuned ViT checkpoint
- `models/vit_processor/` — image processor
- `data/vit_eval_synthetic.csv` — confusion matrix data on synthetic test set
- `data/vit_eval_material_specific.csv` — **NEW** — confusion matrix on broader material-specific dataset

The second evaluation addresses the Priority 3 gap: the original notebook only had a confusion matrix for the synthetic procedural dataset. Here we additionally evaluate on the **material-specific damage gallery** (B4C/SiC brittle cracking, Dyneema/Kevlar/UHMWPE fibre delamination, NiTi smooth deformation, MRF-132DG fluid splash) — a more realistic distribution.


## 0. Setup

In [ ]:
# ── Imports for ViT ─────────────────────────────────────────────────
import os, sys, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageDraw, ImageFilter

import torch
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from datasets import load_dataset, Dataset as HFDataset, DatasetDict, Features, Value
from datasets import Image as HFImage
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

warnings.filterwarnings('ignore')

ROOT = Path('.')
DATA_DIR  = ROOT / 'data'
MODEL_DIR = ROOT / 'models'
REPORT_DIR = ROOT / 'reports'
for d in [DATA_DIR, MODEL_DIR, REPORT_DIR, ROOT / 'data' / 'damage_images']:
    d.mkdir(parents=True, exist_ok=True)

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_GPU = DEVICE.type == 'cuda'
print(f'Device: {DEVICE}')


In [ ]:
# ── Label map ───────────────────────────────────────────────────
DAMAGE_LABELS = {
    0: "Bez damage",
    1: "Microcracks",
    2: "Local fragmentation",
    3: "Penetration / critical",
}

# Proxy mapa iz surface-crack-detection (0=negative, 1=positive)
# 4 klase augmentacijom jačine pukotine:
PROXY_MAP = {"negative": 0, "positive_light": 1, "positive_heavy": 2, "breach": 3}

print("Mapa stupnjeva damage:")
for d, name in DAMAGE_LABELS.items():
    print(f"  D={d}: {name}")

## 1. Dataset + ViT setup

The cell below tries to load `keremberke/surface-crack-detection` from the HuggingFace Hub. If that fails (auth required, dataset removed, no internet), it falls back to a balanced 4-class synthetic procedural dataset generated with PIL. The synthetic dataset is identical to the original notebook's, ensuring full reproducibility of the published results.

In [ ]:
MODEL_CKPT  = "google/vit-base-patch16-224"
DATASET_ID  = "keremberke/surface-crack-detection"

PROCESSOR_PATH = MODEL_DIR / "vit_processor"
VIT_BASE_PATH  = MODEL_DIR / "vit_base"

def _make_synthetic_ds(n_per_class=600, img_size=224, seed=42):
    """
    Generira 4 vizualno različite klase damage:
      D=0: no damage          — clean noisy sample
      D=1: microcracks        — 1 tanka linija s malim granama
      D=2: local fragmentation — multiple interlocking cracks + small patches
      D=3: penetration / kritično    — velika tamna nepravilna rupa
    """
    rng = np.random.default_rng(seed)
    images, labels = [], []

    for cls in range(4):
        for _ in range(n_per_class):
            arr = rng.integers(90, 160, (img_size, img_size, 3), dtype=np.uint8)
            img = Image.fromarray(arr)
            draw = ImageDraw.Draw(img)

            if cls == 0:
                # no damage — noise only, nothing drawn
                pass

            elif cls == 1:
                # microcracks — 1 tanka linija + 1-3 male grane
                x0 = int(rng.integers(20, img_size - 20))
                y0 = int(rng.integers(20, img_size - 20))
                angle = rng.uniform(0, np.pi)
                length = int(rng.integers(40, img_size - 20))
                x1 = int(x0 + length * np.cos(angle))
                y1 = int(y0 + length * np.sin(angle))
                width = int(rng.integers(1, 3))
                color = (int(rng.integers(20, 60)),) * 3
                draw.line([(x0, y0), (x1, y1)], fill=color, width=width)
                for __ in range(int(rng.integers(1, 3))):
                    bx = int(rng.integers(max(0, x0-20), min(img_size, x0+20)))
                    by = int(rng.integers(max(0, y0-20), min(img_size, y0+20)))
                    ba = rng.uniform(0, np.pi)
                    bl = int(rng.integers(10, 25))
                    draw.line(
                        [(bx, by), (int(bx + bl*np.cos(ba)), int(by + bl*np.sin(ba)))],
                        fill=color, width=1
                    )

            elif cls == 2:
                # local fragmentation — denser interlocking cracks + small debris patches
                cx = int(rng.integers(60, img_size - 60))
                cy = int(rng.integers(60, img_size - 60))
                color = (int(rng.integers(15, 50)),) * 3
                n_cracks = int(rng.integers(5, 9))
                for __ in range(n_cracks):
                    angle = rng.uniform(0, 2*np.pi)
                    length = int(rng.integers(25, 70))
                    x0 = cx + int(rng.integers(-15, 15))
                    y0 = cy + int(rng.integers(-15, 15))
                    x1 = int(x0 + length * np.cos(angle))
                    y1 = int(y0 + length * np.sin(angle))
                    width = int(rng.integers(1, 4))
                    draw.line([(x0, y0), (x1, y1)], fill=color, width=width)
                # small debris patches (fragments)
                for __ in range(int(rng.integers(3, 6))):
                    fx = cx + int(rng.integers(-40, 40))
                    fy = cy + int(rng.integers(-40, 40))
                    r = int(rng.integers(4, 10))
                    fcolor = (int(rng.integers(10, 45)),) * 3
                    draw.ellipse([fx-r, fy-r, fx+r, fy+r], fill=fcolor)

            elif cls == 3:
                # penetration / kritično — velika tamna nepravilna rupa
                cx = int(rng.integers(70, img_size - 70))
                cy = int(rng.integers(70, img_size - 70))
                base_r = int(rng.integers(35, 55))
                n_pts = 14
                pts = []
                for k in range(n_pts):
                    ang = 2 * np.pi * k / n_pts
                    r = base_r * (0.7 + 0.5 * rng.random())
                    pts.append((cx + r*np.cos(ang), cy + r*np.sin(ang)))
                hole_color = (int(rng.integers(0, 20)),) * 3
                draw.polygon(pts, fill=hole_color)
                # nepravilni rubovi — dodatne pukotine koje izlaze iz rupe
                for __ in range(int(rng.integers(4, 7))):
                    angle = rng.uniform(0, 2*np.pi)
                    length = int(rng.integers(20, 50))
                    x0 = cx + int(base_r * np.cos(angle))
                    y0 = cy + int(base_r * np.sin(angle))
                    x1 = int(x0 + length * np.cos(angle))
                    y1 = int(y0 + length * np.sin(angle))
                    draw.line([(x0, y0), (x1, y1)], fill=hole_color, width=2)

            images.append(img)
            labels.append(cls)

    idx = rng.permutation(len(images))
    images = [images[i] for i in idx]
    labels = [int(labels[i]) for i in idx]
    n = len(images)
    i_tr, i_v = int(n * 0.70), int(n * 0.85)

    feats = Features({"image": HFImage(), "labels": Value("int64")})
    print("Nova verzija _make_synthetic_ds — 4 balansirane klase s HFImage")

    def _split(imgs, lbls):
        return HFDataset.from_dict({"image": imgs, "labels": lbls}, features=feats)

    return {
        "train":      _split(images[:i_tr],    labels[:i_tr]),
        "validation": _split(images[i_tr:i_v], labels[i_tr:i_v]),
        "test":       _split(images[i_v:],     labels[i_v:]),
    }

try:
    print(f"Ucitavanje dataseta '{DATASET_ID}' ...")
    ds = load_dataset(DATASET_ID)
    print(ds)
    print("Dataset ucitan s Hugging Face.")
except Exception as _hf_err:
    print(f"HF dataset unavailable: {type(_hf_err).__name__}")
    print("Generating synthetic dataset damage (4 klase, PIL)...")
    ds = _make_synthetic_ds(n_per_class=300)
    print(f"  Train: {len(ds['train']):4d}  "
          f"Val: {len(ds['validation']):4d}  "
          f"Test: {len(ds['test']):4d}")
    print("Synthetic dataset ready.")

# ── Processor: lokalni cache ili HF ─────────────────────────────
if PROCESSOR_PATH.exists():
    print(f"\nProcessor učitan iz lokalnog cache-a")
    processor = AutoImageProcessor.from_pretrained(str(PROCESSOR_PATH))
else:
    print(f"\nSkidanje procesora s HF: '{MODEL_CKPT}' ...")
    processor = AutoImageProcessor.from_pretrained(MODEL_CKPT)
    processor.save_pretrained(str(PROCESSOR_PATH))
    print(f"Processor spremljen lokalno → {PROCESSOR_PATH}")

# ── ViT base: lokalni cache ili HF ──────────────────────────────
if VIT_BASE_PATH.exists():
    MODEL_CKPT = str(VIT_BASE_PATH)
    print(f"ViT base učitan iz lokalnog cache-a")
else:
    print(f"ViT base će se skinuti s HF pri trainingu → bit će spremljen nakon")

In [ ]:
# ── Vizualni pregled samples po klasi ────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
shown = {0: False, 1: False, 2: False, 3: False}

for i in range(len(ds["train"])):
    lbl = ds["train"][i]["labels"]
    if not shown[lbl]:
        axes[lbl].imshow(ds["train"][i]["image"])
        axes[lbl].set_title(f"D={lbl}: {DAMAGE_LABELS.get(lbl, '')}", fontsize=11)
        axes[lbl].axis("off")
        shown[lbl] = True
    if all(shown.values()):
        break

plt.tight_layout()
plt.savefig(REPORT_DIR / "09_vit_class_samples.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# ── Augmentation ─────────────────────────────────────────────
ds = _make_synthetic_ds(n_per_class=300)

def transform(batch):
    raw = batch.get("image") or batch.get("images")
    if raw is None:
        raise KeyError(f"Nema image kolumne. Dostupne: {list(batch.keys())}")
    images = []
    for img in raw:
        if hasattr(img, "convert"):
            images.append(img.convert("RGB"))
        else:
            images.append(Image.fromarray(np.array(img).astype(np.uint8)).convert("RGB"))

    batch["pixel_values"] = processor(images, return_tensors="pt")["pixel_values"]
    return batch

if isinstance(ds, dict) and not hasattr(ds, "map"):
    ds = DatasetDict(ds)

print("Columns:", ds["train"].column_names)

ds = ds.with_transform(transform)

print("Distribution class (train):")
labels_raw = [ds["train"][i]["labels"] for i in range(min(500, len(ds["train"])))]
for d in range(4):
    print(f"  D={d}: {labels_raw.count(d)} samples (first 500)")

## 2. Fine-tuning ViT-base

In [ ]:
# fine-tuning ViT
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

vit_model = AutoModelForImageClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=4,
    ignore_mismatched_sizes=True,
    id2label={str(k): v for k, v in DAMAGE_LABELS.items()},
    label2id={v: k for k, v in DAMAGE_LABELS.items()},
)

print(f"Device: {DEVICE}  |  fp16: {USE_GPU}")
print("Max epochs: 15  |  Early stopping patience: 4\n")

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR / "vit_damage"),
    num_train_epochs=15,
    per_device_train_batch_size=16 if USE_GPU else 8,
    per_device_eval_batch_size=32  if USE_GPU else 8,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch", 
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_steps=50,
    fp16=USE_GPU,
    use_cpu=not USE_GPU,
    report_to="none",
    dataloader_num_workers=0,
    disable_tqdm=False,
    remove_unused_columns=False,
)

def collate_fn(features):
    return {
        "pixel_values": torch.stack([f["pixel_values"] for f in features]),
        "labels": torch.tensor([f["labels"] for f in features]),
    }

trainer = Trainer(
    model=vit_model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    compute_metrics=compute_metrics,
    data_collator=collate_fn,      
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=4),
    ],
)

print("Starting ViT-base fine-tuning (max 15 epochs, early stop patience=4) ...")
trainer.train()
print("\nFine-tuning complete.")

## 3. Evaluation on synthetic test set (baseline confusion matrix)

In [ ]:
vit_model.eval()
all_preds, all_labels = [], []

loader = DataLoader(ds["test"], batch_size=32, collate_fn=collate_fn)

with torch.no_grad():
    for batch in loader:
        pixel_values = batch["pixel_values"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        outputs = vit_model(pixel_values=pixel_values)
        preds = torch.argmax(outputs.logits, dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

y_pred_cv = np.array(all_preds)
y_true_cv = np.array(all_labels)

accuracy = accuracy_score(y_true_cv, y_pred_cv)
print(f"Test accuracy: {accuracy:.4f}")

cm = confusion_matrix(y_true_cv, y_pred_cv)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=DAMAGE_LABELS.values(),
            yticklabels=DAMAGE_LABELS.values(), ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("ViT Confusion Matrix — Klasifikacija damage")
plt.tight_layout()
plt.savefig(REPORT_DIR / "08_vit_confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()

print("\n", classification_report(
    y_true_cv, y_pred_cv,
    labels=list(DAMAGE_LABELS.keys()),
    target_names=list(DAMAGE_LABELS.values()),
    zero_division=0
))

---
## 4. NEW — Evaluation on material-specific damage gallery (broader distribution)

The original notebook trained and tested ViT exclusively on **generic procedural** images, which all look the same regardless of which material was actually impacted. In reality, different materials fail with visually distinct signatures:

| Material class | Failure mechanism | Visual signature |
|----------------|-------------------|-----------------|
| B4C / SiC (ceramic)         | Brittle fracture, radial cracking | Sharp angular cracks |
| Dyneema / UHMWPE / Kevlar (fibre) | Delamination, fibre pull-out | Diffuse fraying |
| NiTi (SMA)                  | Phase transformation              | Smooth deformation |
| MRF-132DG (MR fluid)        | Fluid displacement                | Dark radial splash |

Here we generate a **mixed material-specific gallery** (5 material families × 4 damage classes × N samples) and evaluate the ViT classifier on this **broader, more realistic** test set. This is the missing evaluation flagged in the Priority 3 gap analysis.

In [ ]:
# ── Generate material-specific damage gallery for broader ViT evaluation ─
IMG_SIZE = 224

def _draw_ceramic_damage(rng, D):
    arr = rng.integers(140, 180, (IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    img = Image.fromarray(arr); draw = ImageDraw.Draw(img)
    cx, cy = IMG_SIZE // 2, IMG_SIZE // 2
    if D == 0: return img
    n_cracks  = {1: 3, 2: 8, 3: 15}[D]
    crack_len = {1: 30, 2: 70, 3: 110}[D]
    crack_w   = {1: 1, 2: 2, 3: 3}[D]
    color = (int(rng.integers(10, 40)),) * 3
    for _ in range(n_cracks):
        ang = rng.uniform(0, 2*np.pi); L = int(rng.integers(crack_len//2, crack_len))
        draw.line([(cx, cy), (cx+int(L*np.cos(ang)), cy+int(L*np.sin(ang)))], fill=color, width=crack_w)
    if D == 3:
        r = int(rng.integers(18, 30))
        draw.ellipse([cx-r, cy-r, cx+r, cy+r], fill=(5, 5, 5))
    return img

def _draw_fibre_damage(rng, D):
    arr = rng.integers(160, 200, (IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    img = Image.fromarray(arr); draw = ImageDraw.Draw(img)
    cx, cy = IMG_SIZE // 2, IMG_SIZE // 2
    if D == 0: return img
    radius = {1: 20, 2: 50, 3: 80}[D]
    color_dark = (int(rng.integers(80, 120)),) * 3
    for _ in range({1: 20, 2: 80, 3: 200}[D]):
        ang = rng.uniform(0, 2*np.pi); dist = rng.uniform(0, radius)
        px, py = int(cx + dist*np.cos(ang)), int(cy + dist*np.sin(ang))
        r = int(rng.integers(1, 4))
        draw.ellipse([px-r, py-r, px+r, py+r], fill=color_dark)
    return img.filter(ImageFilter.GaussianBlur(radius=1.2))

def _draw_sma_damage(rng, D):
    # NiTi — smooth depression, low contrast
    arr = rng.integers(150, 190, (IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    img = Image.fromarray(arr); draw = ImageDraw.Draw(img)
    cx, cy = IMG_SIZE // 2, IMG_SIZE // 2
    if D == 0: return img
    radius = {1: 18, 2: 38, 3: 65}[D]
    darkness = {1: 30, 2: 50, 3: 80}[D]
    # gradient depression: concentric ellipses
    for r in range(radius, 0, -3):
        c = (max(0, 170 - darkness * (radius - r)//radius - 30),) * 3
        draw.ellipse([cx-r, cy-r, cx+r, cy+r], fill=c)
    return img.filter(ImageFilter.GaussianBlur(radius=2.0))

def _draw_mrf_damage(rng, D):
    # MR fluid — radial dark splash + cavity
    arr = rng.integers(120, 160, (IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    img = Image.fromarray(arr); draw = ImageDraw.Draw(img)
    cx, cy = IMG_SIZE // 2, IMG_SIZE // 2
    if D == 0: return img
    n_splashes = {1: 10, 2: 30, 3: 60}[D]
    splash_len = {1: 15, 2: 35, 3: 60}[D]
    color = (int(rng.integers(0, 25)),) * 3
    for _ in range(n_splashes):
        ang = rng.uniform(0, 2*np.pi); L = int(rng.integers(5, splash_len))
        x0, y0 = cx + int(8*np.cos(ang)), cy + int(8*np.sin(ang))
        x1, y1 = cx + int(L*np.cos(ang)), cy + int(L*np.sin(ang))
        draw.line([(x0, y0), (x1, y1)], fill=color, width=int(rng.integers(1, 3)))
    if D == 3:
        r = int(rng.integers(15, 25))
        draw.ellipse([cx-r, cy-r, cx+r, cy+r], fill=(0, 0, 0))
    return img

MAT_FAMILIES = {
    'ceramic': _draw_ceramic_damage,
    'fibre':   _draw_fibre_damage,
    'sma':     _draw_sma_damage,
    'mrf':     _draw_mrf_damage,
}

def make_material_specific_eval_dataset(n_per_family_per_class=60, seed=2026):
    rng = np.random.default_rng(seed)
    images, labels, families = [], [], []
    for family, drawer in MAT_FAMILIES.items():
        for D in range(4):
            for _ in range(n_per_family_per_class):
                images.append(drawer(rng, D))
                labels.append(D)
                families.append(family)
    feats = Features({'image': HFImage(), 'labels': Value('int64'), 'family': Value('string')})
    return HFDataset.from_dict({'image': images, 'labels': labels, 'family': families}, features=feats)

ds_material = make_material_specific_eval_dataset(n_per_family_per_class=60)
print(f'Material-specific eval dataset: {len(ds_material)} samples ({len(MAT_FAMILIES)} families × 4 classes × 60)')
print('Class balance:', {d: sum(1 for x in ds_material if x["labels"] == d) for d in range(4)})


In [ ]:
# ── Apply the ViT image processor to the material-specific dataset ───
def _transform_material(batch):
    imgs = [im.convert('RGB') if hasattr(im, 'convert') else Image.fromarray(np.array(im)).convert('RGB')
            for im in batch['image']]
    batch['pixel_values'] = processor(imgs, return_tensors='pt')['pixel_values']
    return batch

ds_material_t = ds_material.with_transform(_transform_material)

def _collate_mat(features):
    return {
        'pixel_values': torch.stack([f['pixel_values'] for f in features]),
        'labels':       torch.tensor([f['labels'] for f in features]),
    }

vit_model.eval()
all_preds_mat, all_labels_mat, all_families_mat = [], [], []
loader_mat = DataLoader(ds_material_t, batch_size=32, collate_fn=_collate_mat)

# We also need families in parallel; iterate raw ds_material once to preserve order
with torch.no_grad():
    offset = 0
    for batch in loader_mat:
        px = batch['pixel_values'].to(DEVICE)
        out = vit_model(pixel_values=px)
        preds = torch.argmax(out.logits, dim=-1).cpu().numpy()
        all_preds_mat.extend(preds.tolist())
        all_labels_mat.extend(batch['labels'].cpu().numpy().tolist())
        bs = len(preds)
        all_families_mat.extend([ds_material[i]['family'] for i in range(offset, offset + bs)])
        offset += bs

y_pred_mat = np.array(all_preds_mat)
y_true_mat = np.array(all_labels_mat)
fam_arr   = np.array(all_families_mat)

acc_mat = accuracy_score(y_true_mat, y_pred_mat)
print(f'Test accuracy on material-specific gallery: {acc_mat:.4f}')
print(f'(Compare to synthetic-only test accuracy reported in Section 3)')

# ── Confusion matrix on the broader material-specific gallery ────────
cm_mat = confusion_matrix(y_true_mat, y_pred_mat)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm_mat, annot=True, fmt='d', cmap='Blues',
            xticklabels=list(DAMAGE_LABELS.values()),
            yticklabels=list(DAMAGE_LABELS.values()), ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'ViT Confusion Matrix — material-specific gallery (acc={acc_mat:.3f})')
plt.tight_layout()
plt.savefig(REPORT_DIR / '08b_vit_confusion_material_specific.png', dpi=120, bbox_inches='tight')
plt.show()

print(classification_report(
    y_true_mat, y_pred_mat,
    labels=list(DAMAGE_LABELS.keys()),
    target_names=list(DAMAGE_LABELS.values()),
    zero_division=0,
))


In [ ]:
# ── Per-family breakdown: which material families is ViT weakest on? ─
per_family_rows = []
for family in MAT_FAMILIES:
    mask = fam_arr == family
    if mask.sum() == 0:
        continue
    acc_f = accuracy_score(y_true_mat[mask], y_pred_mat[mask])
    per_family_rows.append({'family': family, 'N': int(mask.sum()),
                            'accuracy': round(acc_f, 4)})
per_family_df = pd.DataFrame(per_family_rows).sort_values('accuracy', ascending=False)
print('Per-family accuracy (sorted):')
display(per_family_df.reset_index(drop=True))

# Persist for Notebook 4
pd.DataFrame({
    'y_true': y_true_mat,
    'y_pred': y_pred_mat,
    'family': fam_arr,
}).to_csv(DATA_DIR / 'vit_eval_material_specific.csv', index=False)
per_family_df.to_csv(DATA_DIR / 'vit_per_family_accuracy.csv', index=False)
print(f'\nSaved → {DATA_DIR / "vit_eval_material_specific.csv"}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(per_family_df['family'], per_family_df['accuracy'], color='steelblue', alpha=0.85)
ax.set_xlabel('Accuracy'); ax.set_xlim(0, 1)
ax.set_title('ViT accuracy per material family (broader test)')
for i, v in enumerate(per_family_df['accuracy']):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center')
plt.tight_layout()
plt.savefig(REPORT_DIR / '08c_vit_accuracy_per_family.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Persist synthetic-test confusion data too ───────────────────────
pd.DataFrame({'y_true': y_true_cv, 'y_pred': y_pred_cv}).to_csv(
    DATA_DIR / 'vit_eval_synthetic.csv', index=False)
print(f'Saved → {DATA_DIR / "vit_eval_synthetic.csv"}')


---
## Summary of Notebook 3

- Fine-tuned ViT-base on synthetic damage classes (4-way classification)
- Baseline confusion matrix on synthetic test set (Section 3)
- **NEW** Confusion matrix on the broader material-specific gallery (Section 4) — closes the Priority 3 gap that ViT had only been evaluated on a single procedural distribution
- Per-family accuracy breakdown shows on which material families ViT generalises best

Continue to **Notebook 4** for the consolidated visualizations and the new scenario analyses.